In [7]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output 
import pandas as pd
import plotly.express as px

# Load the data using pandas 
data = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv")

# Create a dash application
app = dash.Dash(__name__)

# Create the dropdown menu options
dropdown_options = [
    {'label': 'All Sites', 'value': 'ALL'},
    {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'},
    {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'},
    {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'},
    {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}
]

# Create the layout of the dashboard
app.layout = html.Div([
    # Add an H1 header to the dashboard for the title
    html.H1("SpaceX Launch Records Dashboard", style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    
    # Add a dropdown list to enable Launch Site selection
    html.Div([ 
        html.Label("Select Launch Site:"),
        dcc.Dropdown(id='site-dropdown',
                     options=dropdown_options,
                     value="ALL",
                     placeholder="Select a Launch Site here",
                     style={'marginRight': '2em'},
                     searchable=True)
    ]),
    html.Br(),

    # TASK 2: Add a pie chart to show the total successful launches count for all sites or selected site
    html.Div(dcc.Graph(id='success-pie-chart')),
    html.Br(),

    # TASK 3: Add the range slider to select the payload range
    html.P("Payload range (Kg):"),
    dcc.RangeSlider(id='payload-slider',
                    min=0, max=10000, step=1000,
                    marks={0: '0', 2500: '2500', 5000: '5000', 7500: '7500', 10000: '10000'},
                    value=[0, 10000]),
    html.Br(),

    # TASK 4: Add a scatter chart to show the correlation between payload and launch success
    html.Div(dcc.Graph(id='success-payload-scatter-chart')),
])

# Callback for the pie chart
@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def get_pie_chart(entered_site):
    if entered_site == 'ALL':
        fig = px.pie(data, values='class', 
                     names='Launch Site',
                     title='Total Success Launches by Site')
        return fig
    else:
        filtered_df = data[data['Launch Site'] == entered_site]
        # For a single site, we want to see the count of success (1) vs failure (0)
        class_counts = filtered_df['class'].value_counts().reset_index()
        class_counts.columns = ['class', 'count']
        fig = px.pie(class_counts, values='count', names='class', 
                     title=f'Total Success Launches for site {entered_site}')
        return fig

# Callback for the scatter plot
@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [Input(component_id='site-dropdown', component_property='value'),
     Input(component_id='payload-slider', component_property='value')]
)
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range
    # Filter by payload limits first
    mask = (data['Payload Mass (kg)'] >= low) & (data['Payload Mass (kg)'] <= high)
    filtered_df = data[mask]
    
    if entered_site == 'ALL':
        fig = px.scatter(filtered_df, x='Payload Mass (kg)', y='class', 
                         color='Booster Version Category',
                         title='Correlation between Payload and Success for all Sites')
        return fig
    else:
        # Further filter by specific site
        site_df = filtered_df[filtered_df['Launch Site'] == entered_site]
        fig = px.scatter(site_df, x='Payload Mass (kg)', y='class', 
                         color='Booster Version Category',
                         title=f'Correlation between Payload and Success for site {entered_site}')
        return fig

# Run the app
if __name__ == '__main__':
    # Change app.run_server(debug=True) to:
    app.run(debug=True)